In [1]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
from tqdm import tqdm
import warnings
import re 

# PyCaret의 plot_model 및 create_model을 사용하기 위해 임포트 추가
from pycaret.regression import setup, compare_models, predict_model, create_model, plot_model 
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")


class MercariPyCaretAnalyzer:
    """
    Mercari Price Suggestion Challenge용 PyCaret 분석기 (GPU 학습 옵션 추가)
    """

    def __init__(
        self,
        data_dir="../data",
        images_dir="../images",
        results_dir="../results",
    ):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        self.train = None
        self.test = None
        self.best_model = None
        self.setup_result = None
        self.metrics = {}

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # 🔹 (공통 유틸) 희귀 카테고리/브랜드를 "Other" 그룹으로 묶는 함수
    # (기존 코드와 동일)
    # ------------------------------------------------------------------
    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        combined = pd.concat([self.train[col], self.test[col]], axis=0)
        value_counts = combined.value_counts()
        top_values = value_counts.index[:top_k]

        self.train[col] = self.train[col].where(
            self.train[col].isin(top_values), rare_label
        )
        self.test[col] = self.test[col].where(
            self.test[col].isin(top_values), rare_label
        )

    # ------------------------------------------------------------------
    # 🔹 (공통 유틸) 텍스트 간단 정규화 함수
    # (기존 코드와 동일)
    # ------------------------------------------------------------------
    def _simple_normalize(self, text: str) -> str:
        text = str(text).lower()
        text = re.sub(r"[_\-\./]", " ", text)
        text = re.sub(r"\d+", " num ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    # ------------------------------------------------------------------
    # 1. 데이터 로딩 + 기본 전처리 + 희귀 카테고리/브랜드 통합 + 길이 피처 생성
    # (기존 코드와 동일)
    # ------------------------------------------------------------------
    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t"):
        print("📂 데이터 로딩 시작...")
        train_path = os.path.join(self.data_dir, train_file)
        test_path = os.path.join(self.data_dir, test_file)

        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        print(f"original data shape : train {self.train.shape}, test {self.test.shape}")
        print("✅ Price 외 결측치 처리 및 데이터 전처리 시작...")

        # 1-1) price 0 제거 + NaN 제거 (train만 해당)
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        
        # 1-2) category_name → main_cat / sub_cat / sub_sub_cat 분해, 텍스트 결측치 채우기
        for df_name, df in [("train", self.train), ("test", self.test)]:
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (
                        x.split("/")
                        if isinstance(x, str) and "/" in x
                        else ["missing", "missing", "missing"]
                    )
                )
            )
            df["brand_name"] = df["brand_name"].fillna("Unknown").astype(str)
            df["category_name"] = df["category_name"].fillna("Unknown").astype(str)
            df["item_description"] = (
                df["item_description"].fillna("No description").astype(str)
            )
            df["name"] = df["name"].fillna("No name").astype(str)

            if df_name == "train":
                self.train = df.reset_index(drop=True)
            else:
                self.test = df.reset_index(drop=True)

        # 1-3) price 로그 변환 (log1p)
        self.train["price"] = np.log1p(self.train["price"])

        # 1-4) 희귀 브랜드 / 카테고리 통합
        print("📊 희귀 카테고리/브랜드 통합(rare category collapsing) 시작...")
        self._collapse_rare_values("brand_name", top_k=4500, rare_label="Other_brand")
        self._collapse_rare_values("main_cat", top_k=1000, rare_label="Other_main")
        self._collapse_rare_values("sub_cat", top_k=1000, rare_label="Other_sub")
        self._collapse_rare_values(
            "sub_sub_cat", top_k=1000, rare_label="Other_sub_sub"
        )

        # 1-5) 텍스트 길이 기반 수치 피처 추가
        for df_name, df in [("train", self.train), ("test", self.test)]:
            df["name_len_char"] = df["name"].astype(str).str.len()
            df["name_len_word"] = df["name"].astype(str).str.split().str.len()
            df["desc_len_char"] = df["item_description"].astype(str).str.len()
            df["desc_len_word"] = (
                df["item_description"].astype(str).str.split().str.len()
            )

        # 1-6) shipping / item_condition_id를 category 타입으로 캐스팅
        for df in [self.train, self.test]:
            df["shipping"] = df["shipping"].astype("category")
            df["item_condition_id"] = df["item_condition_id"].astype("category")

        print(
            f"\n✅ 데이터 로드 완료: train {self.train.shape}, test {self.test.shape}"
        )

    # ------------------------------------------------------------------
    # 2. 텍스트 벡터화 + 차원 축소 + 카테고리/길이 피처 결합
    # (기존 코드와 동일)
    # ------------------------------------------------------------------
    def vectorize_text(
        self,
        text_columns=["name", "item_description"],
        method="tfidf",
        max_features=50000,
        n_components=100,
    ):
        print("📝 텍스트 벡터화 및 차원 축소 시작...")

        # 2-1) 텍스트 정규화 컬럼(*_clean) 생성
        for col in text_columns:
            clean_col = f"{col}_clean"
            if clean_col not in self.train.columns:
                self.train[clean_col] = self.train[col].astype(str).apply(
                    self._simple_normalize
                )
                self.test[clean_col] = self.test[col].astype(str).apply(
                    self._simple_normalize
                )

        vectors = []
        feature_names = []

        # 2-2) 텍스트 컬럼별로 TF-IDF (또는 Count) + SVD 적용
        for col in tqdm(text_columns, desc="Text columns"):
            clean_col = f"{col}_clean"

            if method == "tfidf":
                vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
            elif method == "count":
                vec = CountVectorizer(max_features=max_features, ngram_range=(1, 2))
            else:
                raise ValueError("method must be 'tfidf' or 'count'")

            combined_text = pd.concat(
                [self.train[clean_col], self.test[clean_col]], axis=0
            )
            vec.fit(combined_text)

            train_vec = vec.transform(self.train[clean_col])
            test_vec = vec.transform(self.test[clean_col])

            # 2-3) 차원 축소: TruncatedSVD
            if n_components < train_vec.shape[1]:
                svd = TruncatedSVD(n_components=n_components, random_state=23)
                train_vec = svd.fit_transform(train_vec)
                test_vec = svd.transform(test_vec)
            else:
                train_vec = train_vec.toarray()
                test_vec = test_vec.toarray()

            vectors.append((train_vec, test_vec))
            feature_names.append([f"{col}_{i}" for i in range(train_vec.shape[1])])

            del combined_text, vec
            gc.collect()

        # 2-4) 모든 텍스트 피처를 가로 방향으로 합치기
        train_features = np.hstack([v[0] for v in vectors])
        test_features = np.hstack([v[1] for v in vectors])

        self.train_vectorized = pd.DataFrame(
            train_features, columns=[f for sub in feature_names for f in sub]
        )
        self.test_vectorized = pd.DataFrame(
            test_features, columns=[f for sub in feature_names for f in sub]
        )

        # 2-5) 카테고리/브랜드/배송 + 텍스트 길이 피처를 그대로 붙이기
        categorical_cols = [
            "main_cat", "sub_cat", "sub_sub_cat", "brand_name",
            "item_condition_id", "shipping",
        ]

        numeric_length_cols = [
            "name_len_char", "name_len_word", "desc_len_char", "desc_len_word",
        ]

        for col in categorical_cols + numeric_length_cols:
            if col in self.train.columns:
                self.train_vectorized[col] = (
                    self.train[col].reset_index(drop=True)
                )
                self.test_vectorized[col] = (
                    self.test[col].reset_index(drop=True)
                )

        print(
            f"✅ 벡터화 + 차원 축소 + 카테고리/길이 피처 추가 완료: "
            f"train {self.train_vectorized.shape}, test {self.test_vectorized.shape}"
        )

    # ------------------------------------------------------------------
    # 3. PyCaret setup
    # (기존 코드와 동일)
    # ------------------------------------------------------------------
    def setup_pycaret(self, session_id=23):
        if not hasattr(self, "train_vectorized"):
            raise ValueError("먼저 vectorize_text()를 실행하세요.")

        print("🔧 PyCaret setup 시작...")

        categorical_cols = [
            "main_cat", "sub_cat", "sub_sub_cat", "brand_name",
            "item_condition_id", "shipping",
        ]
        existing_categorical = [
            col for col in categorical_cols if col in self.train_vectorized.columns
        ]

        # price가 이미 log1p로 변환되었으므로 transformation=False
        self.setup_result = setup(
            data=self.train_vectorized.assign(
                price=self.train["price"].reset_index(drop=True)
            ),
            target="price",
            session_id=session_id,
            categorical_features=existing_categorical if existing_categorical else None,
            normalize=True,
            transformation=False,
            verbose=True,
        )
        print("✅ PyCaret setup 완료")

    # ------------------------------------------------------------------
    # 4. Base model 탐색 (Optional: GPU 사용을 위해 아래 함수 사용 권장)
    # ------------------------------------------------------------------
    def find_base_model(self, sort_metric="R2"):
        """
        PyCaret 기본 CPU 모드로 베이스 모델 탐색 (느릴 수 있음)
        """
        if self.setup_result is None:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")

        print("🔍 Base model 탐색 시작 (CPU 모드)...")
        self.best_model = compare_models(sort=sort_metric, n_select=1)
        print(f"🏆 Best model 선택 완료: {self.best_model}")
        return self.best_model

    # ------------------------------------------------------------------
    # 4.5. 💡 GPU 사용 LightGBM 모델 학습 (추가된 기능)
    # ------------------------------------------------------------------
    def train_best_model_with_gpu(self):
        """
        LightGBM 모델을 GPU 학습 옵션으로 생성 및 학습합니다.
        (LightGBM은 GPU 학습을 공식적으로 지원하며, Mercari 데이터셋에 적합합니다.)
        """
        if self.setup_result is None:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")
            
        print("🚀 LightGBM (LGBM) 모델을 GPU 옵션으로 학습 시작...")
        
        try:
            # create_model을 사용하여 GPU 옵션 설정
            gpu_lgbm = create_model(
                'lightgbm', 
                # n_estimators나 learning_rate 등 하이퍼파라미터는 필요에 따라 조정
                n_estimators=3000, 
                learning_rate=0.05, 
                # ✅ GPU 사용을 위한 핵심 파라미터
                device='gpu', 
                gpu_use_dp=False, # 단정밀도 사용 (더 빠름)
                random_state=23, 
                verbose=False
            )
            self.best_model = gpu_lgbm
            print(f"✅ GPU 기반 LightGBM 모델 학습 및 선택 완료: {self.best_model}")
        except Exception as e:
            print(f"❌ LightGBM GPU 학습 실패: {e}")
            print("   PyCaret 환경에서 GPU 옵션 설정 및 라이브러리(CUDA, lightgbm-gpu) 설치를 확인하세요.")
            self.best_model = None
            
        return self.best_model


    # ------------------------------------------------------------------
    # 5. 모델 성능 저장 (원래 가격 스케일에서 R2/RMSE/MAE 계산)
    # (기존 코드의 로직은 유지하되, self.train['price'] 사용 시점 수정)
    # ------------------------------------------------------------------
    def save_metrics(self, metrics_dict=None, model_name=None):
        if metrics_dict is None:
            if self.best_model is None:
                raise ValueError("모델이 없습니다. find_base_model() 또는 train_best_model_with_gpu()를 실행하세요.")

            # train 전체에 대해 예측 수행
            pred_df = predict_model(self.best_model, data=self.train_vectorized.copy())

            # 로그 스케일 타깃/예측
            # self.train["price"]는 이미 log1p 변환된 값
            y_log_true = self.train["price"].values
            y_log_pred = pred_df["Label"].values

            # expm1으로 원래 가격 스케일로 되돌리기
            y_true = np.expm1(y_log_true)
            y_pred = np.expm1(y_log_pred)

            r2 = r2_score(y_true, y_pred)
            rmse = mean_squared_error(y_true, y_pred, squared=False)
            mae = mean_absolute_error(y_true, y_pred)

            metrics_dict = {
                "R2": round(r2, 4),
                "RMSE": round(rmse, 4),
                "MAE": round(mae, 4),
            }

        self.metrics = metrics_dict

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        if model_name is None:
            model_name = str(self.best_model).split("(")[0]
        file_path = os.path.join(
            self.results_dir, f"{model_name}_metrics_{timestamp}.json"
        )

        with open(file_path, "w") as f:
            json.dump(self.metrics, f, indent=4)

        print(f"💾 Metrics 저장 완료: {file_path}")

    # ------------------------------------------------------------------
    # 6. 시각화
    # (기존 코드와 동일)
    # ------------------------------------------------------------------
    def visualize_model(self, plots=["residuals", "feature"]):
        if self.best_model is None:
            raise ValueError("먼저 find_base_model() 또는 train_best_model_with_gpu()로 모델을 선택하세요.")

        print("🎨 시각화 시작...")
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = str(self.best_model).split("(")[0]

        for p in plots:
            try:
                plot_name = "feature" if p == "feature_importance" else p

                save_path = os.path.join(
                    self.images_dir, f"{model_name}_{plot_name}_{timestamp}.png"
                )
                plot_model(self.best_model, plot=plot_name, save=True)
                print(f"✅ {plot_name} plot 저장 완료: {save_path}")
            except Exception as e:
                print(f"⚠️ Plot {p} 실패: {e}")

    # ------------------------------------------------------------------
    # 7. Test 예측 & submission 생성
    # (기존 코드와 동일)
    # ------------------------------------------------------------------
    def predict_test(self, submission_file="submission.csv"):
        if self.best_model is None:
            raise ValueError("먼저 find_base_model() 또는 train_best_model_with_gpu()로 모델을 선택하세요.")

        print("📦 Test 데이터 예측 시작...")

        # test_vectorized에 대해 예측 (로그 스케일)
        predictions = predict_model(self.best_model, data=self.test_vectorized.copy())

        # 로그 예측값을 expm1으로 되돌려 실제 가격 스케일로 변환
        price_log_pred = predictions["Label"].values
        price_pred = np.expm1(price_log_pred)

        submission = pd.DataFrame(
            {"test_id": self.test["test_id"], "price": price_pred}
        )

        submission_path = os.path.join(self.results_dir, submission_file)
        submission.to_csv(submission_path, index=False)
        print(f"💾 Submission 저장 완료: {submission_path}")
        return submission

In [2]:
# 예시 실행 코드 (실제 환경에 맞게 경로와 파일명 조정 필요)
analyzer = MercariPyCaretAnalyzer()


In [3]:
analyzer.load_data(train_file="train.tsv", test_file="test.tsv") 


📂 데이터 로딩 시작...
original data shape : train (1482535, 8), test (693359, 7)
✅ Price 외 결측치 처리 및 데이터 전처리 시작...
📊 희귀 카테고리/브랜드 통합(rare category collapsing) 시작...

✅ 데이터 로드 완료: train (1481661, 15), test (693359, 14)


In [4]:
analyzer.vectorize_text()


📝 텍스트 벡터화 및 차원 축소 시작...


Text columns:  50%|█████     | 1/2 [06:59<06:59, 419.59s/it]


KeyboardInterrupt: 

In [ ]:
analyzer.setup_pycaret()


🔧 PyCaret setup 시작...


,Description,Value
0,Session id,23
1,Target,price
2,Target type,Regression
3,Original data shape,"(1481661, 211)"
4,Transformed data shape,"(1481661, 225)"
5,Transformed train set shape,"(1037162, 225)"
6,Transformed test set shape,"(444499, 225)"
7,Numeric features,204
8,Categorical features,6
9,Preprocess,True


✅ PyCaret setup 완료


In [ ]:

# 💡 GPU 사용 함수 호출
best_gpu_model = analyzer.train_best_model_with_gpu()

if best_gpu_model:
    analyzer.save_metrics()
    analyzer.visualize_model()
    analyzer.predict_test()


🚀 LightGBM (LGBM) 모델을 GPU 옵션으로 학습 시작...
